# Cohort Audit

## Purpose

The goal is to verify participant counts, identifiers,
demographic variables, and basic data quality.It establishes that the metadata are internally consistent

## Data source and expected structure
 The repository describes 225 retained MPI-LEMON participants and four metadata fields.

 Participant IDs are loaded as strings so that their identifier formatting is preserved. Participant-level data remain local; only the aggregate audit summary is saved for the repository.


In [1]:
import pandas as pd

from lemon_connectivity.identifiers import (
    add_canonical_participant_id,
    validate_canonical_participant_ids,
    validate_raw_participant_ids,
)

expected_N = 225
expected_columns = ["sub_id", "age", "sex", "cohort"]
expected_sex = ["female", "male"]
expected_cohort = ["young", "elderly"]
expected_age = [
    "20-25",
    "25-30",
    "30-35",
    "35-40",
    "55-60",
    "60-65",
    "65-70",
    "70-75",
    "75-80",
]

## Load the data

The cohort information file is stored as a TSV file. We load it
into a pandas DataFrame for inspection and analysis.

In [2]:
#Load the TSV data into dataframe
df_chr = pd.read_csv('../data/external/Curvature-FCN-Aging/DATA/cohort_information.tsv', sep='\t', dtype={'sub_id': str})
"Columns match expected:",list(df_chr.columns) == expected_columns

('Columns match expected:', True)

## Structural and missingness audit
##  Initial data inspection

Before performing any transformations, We inspected the shape,
column names, data types, and first few rows of the DataFrame.
The checks below distinguish genuine missing values from blank, whitespace-only, or placeholder text. They also test the exact schema, duplicate rows, duplicate participant IDs, and formatting problems in both headers and values.

In [3]:
def data_quality(df_chr: pd.DataFrame) -> dict:

    report = {}
    # 1. check participants rows 
    report ["total_rows"] = len(df_chr)

    # 2. check for unique counts
    report['unique_subjects'] = df_chr['sub_id'].nunique() if 'sub_id' in df_chr.columns else "NA"

    # 3. check unnamed/index column
    unnamed_col = [col for col in df_chr.columns if "Unnamed:" in str(col)]
    report["has_unnamed_columns"] = len(unnamed_col) > 0 
    report["unnamed_columns_found"] = unnamed_col

    # 4. check leading/trailing space in column names
    trailing_col = [col for col in df_chr.columns if str(col).strip() != str(col)]
    report['has_header_whitespace_issues'] = len(trailing_col) > 0
    report['whitespace_issues_found'] = trailing_col

    # 5. check for missigness
    missing_series = df_chr.replace(r"^\s*$|^(?:NA|null|-)$", pd.NA, regex=True,)
    report['missing_count_per_column'] = missing_series.isna().sum()

    # 6. check for duplication by each column 
    report['duplicate_subject_ids'] = int(df_chr.duplicated(subset=['sub_id']).sum())
    # for categorical data where checking there is no values outside the expected categories.
    report['unique_categories_per_column'] = df_chr[['sex', 'cohort', 'age']].nunique().to_dict()
    # 7. check for shape
    report ["the shape is:"] = df_chr.shape
    # 8. check for columns 
    report["the columns are"] = df_chr.columns
    # 9. check for Duplicates 
    report["number of duplicated:"] = df_chr.duplicated().sum()
    # data type in each columns
    report["data type:"] = df_chr.dtypes

    return pd.Series(report) 
    
data_quality(df_chr)



total_rows                                                                    225
unique_subjects                                                               225
has_unnamed_columns                                                         False
unnamed_columns_found                                                          []
has_header_whitespace_issues                                                False
whitespace_issues_found                                                        []
missing_count_per_column        sub_id    0
age       0
sex       0
cohort    ...
duplicate_subject_ids                                                           0
unique_categories_per_column                    {'sex': 2, 'cohort': 2, 'age': 9}
the shape is:                                                            (225, 4)
the columns are                 Index(['sub_id', 'age', 'sex', 'cohort'], dtyp...
number of duplicated:                                                           0
data type:      

## Initial observations

The cohort information file contains **225 participants** and **four columns**.

The metadata contains four variables:

* `sub_id`: original participant identifier
* `age`: participant age group
* `sex`: participant sex category
* `cohort`: broad participant grouping (`young` or `elderly`)

The DataFrame was inspected for its shape, column names, data types, duplicate rows, and participant identifiers before further analysis.

All original metadata columns are stored as expected, and `sub_id` is loaded as a string to preserve the identifier format.


## Participant identifier validation
The original participant IDs are checked with the reusable validation utility
from `lemon_connectivity.identifiers`. This keeps the same identifier rules in
the cohort audit and later alignment notebooks.

In [4]:
raw_id_report = validate_raw_participant_ids(
    df_chr["sub_id"],
    expected_width=5,
)
raw_id_report


total_ids                               225
missing_ids                               0
non_missing_ids                         225
all_non_missing_ids_numeric            True
all_non_missing_ids_expected_width     True
any_alphabetic_ids                    False
invalid_raw_ids                           0
duplicate_raw_ids                         0
dtype: object

 ### Interpretation

All participant IDs contain exactly five characters and consist only
of numeric digits. No alphabetic characters were found in the original
`sub_id` values.


### Canonical ID creation and Validation

The reusable identifier utility creates a separate `canonical_id` column. It
zero-pads each valid five-digit source identifier to six digits and adds the
`sub-` prefix. The original `sub_id` column remains unchanged.

The canonical IDs are then checked for their prefix, exact format, length,
missingness, and duplication.

In [5]:
df_chr = add_canonical_participant_id(
    df_chr,
    source_column="sub_id",
    output_column="canonical_id",
)

canonical_id_report = validate_canonical_participant_ids(
    df_chr["canonical_id"]
)
canonical_id_report


total_canonical_ids            225
valid_canonical_ids            225
invalid_canonical_ids            0
missing_canonical_ids            0
duplicate_canonical_ids          0
all_match_canonical_format    True
all_have_expected_prefix      True
all_have_expected_length      True
dtype: object

### Interpretation

The canonical participant IDs passed all validation checks.

* All canonical IDs have the expected `sub-` prefix.
* All IDs contain exactly six digits after the prefix.
* All canonical IDs contain exactly 10 characters in total.
* No duplicate canonical IDs were found.
* No missing canonical IDs were found.

Therefore, the `canonical_id` variable follows the expected `sub-XXXXXX` format and can be used as a standardized participant identifier for matching with other datasets.


## Validate demographic categories

Counting unique values is not sufficient: two incorrect labels could still produce the expected number of categories. The observed labels are therefore compared directly with predefined expected sets.

In [6]:
def validate_demographics(df):
    report = {}

    # Check sex
    report["sex_categories"] = df["sex"].unique().tolist()
    report["sex_categories_valid"] = (
        df["sex"].isin(expected_sex).all()
    )

    # Check cohort
    report["cohort_categories"] = df["cohort"].unique().tolist()
    report["cohort_categories_valid"] = (
        df["cohort"].isin(expected_cohort).all()
    )

    # Check age
    report["age_categories"] = df["age"].unique().tolist()
    report["age_categories_valid"] = (
        df["age"].isin(expected_age).all()
    )

    # Counts
    report["sex_counts"] = df["sex"].value_counts().to_dict()
    report["cohort_counts"] = df["cohort"].value_counts().to_dict()
    report["age_counts"] = (
        df["age"].value_counts().sort_index().to_dict()
    )

    return pd.Series(report)
validate_demographics(df_chr)

sex_categories                                                [female, male]
sex_categories_valid                                                    True
cohort_categories                                           [elderly, young]
cohort_categories_valid                                                 True
age_categories             [65-70, 20-25, 25-30, 60-65, 30-35, 70-75, 75-...
age_categories_valid                                                    True
sex_counts                                       {'male': 145, 'female': 80}
cohort_counts                                  {'young': 153, 'elderly': 72}
age_counts                 {'20-25': 79, '25-30': 60, '30-35': 13, '35-40...
dtype: object

### Interpretation

The demographic variables contain only the expected categories. The `sex` variable contains `female` and `male`, while the `cohort` variable contains `young` and `elderly`. The `age` variable contains the nine expected age bands.

The dataset contains 80 female and 145 male participants. The cohort distribution consists of 153 young and 72 elderly participants. The age-group distribution also accounts for all 225 participants.

No unexpected demographic categories were identified.


In [7]:
sex_cohort = pd.crosstab(
    df_chr['cohort'],
    df_chr['sex'],
    margins=True,
    margins_name = "Total"
)

print(sex_cohort)

sex      female  male  Total
cohort                      
elderly      36    36     72
young        44   109    153
Total        80   145    225


### Interpretation

The sex distribution differs between the two cohorts. The young cohort contains `44 female` and `109 male` participants, making it male-dominated. In contrast, the elderly cohort contains `36 female` and `36 male` participants, showing an equal distribution between females and males.

Overall, the sex counts are 80 female and 145 male participants, while the cohort counts are 153 young and 72 elderly participants. These totals account for all 225 participants.


## Age–cohort consistency

The source separates young age bands (20–40 years) from elderly age bands (55–80 years). The following check identifies any participant whose recorded cohort contradicts the expected cohort for their age band.

In [8]:
# age to cohort consistency check
age_to_cohort = pd.crosstab(df_chr['age'], df_chr['cohort'])
print(age_to_cohort)

print()
# mapping the age to specific intervals and check with the cohort
age_mapping = {
    "20-25": "20-40 (Young)",
    "25-30": "20-40 (Young)",
    "30-35": "20-40 (Young)",
    "35-40": "20-40 (Young)",
    "55-60": "55-80 (Elderly)",
    "60-65": "55-80 (Elderly)",
    "65-70": "55-80 (Elderly)",
    "70-75": "55-80 (Elderly)",
    "75-80": "55-80 (Elderly)"
}

age_to_cohort_matrix = age_to_cohort.groupby(age_mapping).sum()
print(age_to_cohort_matrix)



cohort  elderly  young
age                   
20-25         0     79
25-30         0     60
30-35         0     13
35-40         0      1
55-60         4      0
60-65        19      0
65-70        24      0
70-75        22      0
75-80         3      0

cohort           elderly  young
age                            
20-40 (Young)          0    153
55-80 (Elderly)       72      0


### Interpretation

The age–cohort cross-tabulation shows that all participants in the `20–40` age bands belong to the `young` cohort, while all participants in the `55–80` age bands belong to the `elderly` cohort.

After grouping the age bands according to the expected cohort definitions, there were no contradictory age–cohort classifications. The age and cohort variables are therefore internally consistent.

There are no participants in the `40–55` age range, which is consistent with the observed age-band distribution.


In [9]:
data = {# Names of the checks performed
    "What did we check?": [
        "Expected participants",
        "Observed participants",
        "Duplicate participant IDs",
        "Missing participant IDs",
        "Age groups",
        "Female participants",
        "Male participants",
        "Young cohort",
        "Elderly cohort",
        "Duplicate canonical IDs",
        "Missing canonical IDs"
    ],
# Results obtained from the data
    "What did we find?": [
        expected_N,
        len(df_chr),
        df_chr["sub_id"].duplicated().sum(),
        df_chr["sub_id"].isna().sum(),
        df_chr["age"].nunique(),
        (df_chr["sex"] == "female").sum(),
        (df_chr["sex"] == "male").sum(),
        (df_chr["cohort"] == "young").sum(),
        (df_chr["cohort"] == "elderly").sum(),
        df_chr["canonical_id"].duplicated().sum(),
        df_chr["canonical_id"].isna().sum()
    ]
}
# Convert the dictionary into a pandas DataFrame
audit_table = pd.DataFrame(data)
audit_table.to_csv('../results/audit_summary.csv', index=False)
audit_table


,What did we check?,What did we find?
0,Expected participants,225
1,Observed participants,225
2,Duplicate participant IDs,0
3,Missing participant IDs,0
4,Age groups,9
5,Female participants,80
6,Male participants,145
7,Young cohort,153
8,Elderly cohort,72
9,Duplicate canonical IDs,0


### Audit summary interpretation

The audit summary confirms that all 225 expected participants are represented. No duplicate or missing participant IDs were detected, and nine expected age groups were observed. The demographic counts account for all participants, and the canonical identifiers contain no duplicates or missing values.

The audit summary therefore provides a reproducible record of the main metadata quality checks completed in this notebook.


## Final audit conclusion

The cohort metadata audit identified 225 participants across four metadata variables: `sub_id`, `age`, `sex`, and `cohort`. No duplicate participant IDs or missing participant IDs were detected, and the complete rows were unique.

All original participant identifiers contained five numeric digits and were successfully converted into unique canonical identifiers without missingness or duplication.

The demographic variables contained only the expected age bands, sex categories, and cohort labels. The age and cohort classifications were internally consistent, with participants aged 20–40 classified as young and participants aged 55–80 classified as elderly.

The dataset contains 80 female and 145 male participants. The young cohort contains 153 participants and is male-dominated, while the elderly cohort contains 72 participants with an equal female-to-male distribution.

Overall, the metadata audit passed the defined structural, identifier, demographic, and consistency checks. The observed sex imbalance between cohorts and the absence of participants in the 40–55 age range should be considered in subsequent analyses.
